# Code pour structurer la liste d'adjacence du réseau de suivi OSOS

In [2]:
from os import path as os_path
import os

In [3]:
import json
import pandas as pd

In [4]:
git_dir = "/home/maya/chaosscon_fosdem20-research-project"
usercomplete_path = os_path.join(git_dir, 'twarc_data', 'userscomplete.jsonl')

In [5]:
with open(usercomplete_path) as json_file:
    json_data = [json.loads(line) for line in json_file]

In [6]:
json_data[0]

{'id': 953304055337668608,
 'id_str': '953304055337668608',
 'name': 'CHAOSS',
 'screen_name': 'CHAOSSproj',
 'location': '',
 'description': 'CHAOSS Project',
 'url': 'https://t.co/Fe8BzMhMoR',
 'entities': {'url': {'urls': [{'url': 'https://t.co/Fe8BzMhMoR',
     'expanded_url': 'http://chaoss.community',
     'display_url': 'chaoss.community',
     'indices': [0, 23]}]},
  'description': {'urls': []}},
 'protected': False,
 'followers_count': 462,
 'friends_count': 6,
 'listed_count': 9,
 'created_at': 'Tue Jan 16 16:32:53 +0000 2018',
 'favourites_count': 644,
 'utc_offset': None,
 'time_zone': None,
 'geo_enabled': False,
 'verified': False,
 'statuses_count': 667,
 'lang': None,
 'status': {'created_at': 'Tue Mar 17 19:33:09 +0000 2020',
  'id': 1239998223349174273,
  'id_str': '1239998223349174273',
  'full_text': 'RTing to spreading the question https://t.co/bJoqvHRwn7',
  'truncated': False,
  'display_text_range': [0, 31],
  'entities': {'hashtags': [],
   'symbols': [],
   '

In [7]:
quali_path = os_path.join(git_dir, 'Project_Files', '85noms_enrichi.csv')
quali_join_key = 'Twitter_handle'
quali_df = pd.read_csv(quali_path, index_col=quali_join_key)
quali_df

,INDEX,Communauté,Conférence,Role1,Role2,Type,Nom_compte,Abonnements,Abonnés,Tweets_totale_période,Date_début,Date_fin,Retweets,Likes,Mentions,#CHAOSSCon,#FOSDEM,#FOSDEM2020,Autre#
Twitter_handle,,,,,,,,,,,,,,,,,,,
CHAOSSproj,1,Open source,CHAOSS,organisateur,NaN,institution,CHAOSS Project,6,440,290.0,03/10/19,21/02/20,NaN,NaN,NaN,#CHAOSSCon,na,na,NaN
GeorgLink,2,Open source,CHAOSS,speaker,NaN,individu,Georg Link,869,392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
isaferreira_57,3,Open source,CHAOSS,speaker,NaN,individu,Isabella Vieira Ferreira,312,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
johanlinaker,4,Open source,CHAOSS,speaker,NaN,individu,Johan Linåker,342,303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
jsmanrique,5,Open source,CHAOSS,speaker,NaN,individu,Jose Manrique Lopez de la Fuente,1242,2243,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
paulanomalie,81,Open science,FOSDEM,live tweeter,NaN,individu,Paul Girard,223,560,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Celyagd,82,Open science,FOSDEM,live tweeter,NaN,individu,Celya Gruson-Daniel,3136,3458,10.0,31/01/20,01/02/20,NaN,NaN,NaN,na,#FOSDEM,#FOSDEM2020,NaN
Violetilla,83,Open science,FOSDEM,live tweeter,NaN,individu,Violeta,400,217,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Créer les fichiers de liens et de noeuds pour le réseau des friends

In [8]:
friend_name_per_id = {} #créer un dictionnaire des paires de id:screen_name
#créer un nouveau fichier csv destiné à contenir la liste d'adjacence des liens
with open(os_path.join(git_dir, 'Project_Files', 'friends_edges.csv'), 'w') as friends_edges_file :
    for user in json_data:
        user_id = user['id_str']
        twitter_handle = user['screen_name']
        #créer une variable avec l'information qui sert à ordonner les utilisateurs
        try:
            id_quali = quali_df.loc[twitter_handle, 'INDEX'] 
        except:
            print("Twitter id not found in quali file:", twitter_handle)#indiquer les données manquantes
        if id_quali.size != 1:
            print("Skipping duplicated user:", id_quali.index[0]) #indiquer les doublons
            continue
        assert id_quali > 0 and id_quali <= 85
        #ouvrir les fichiers en fonction de leur identifiant quali
        friends_file_name = 'users_friend_ids{}.jsonl'.format(id_quali)
        friends_path = os_path.join(git_dir, 'twarc_data','friends', 'users_friend_ids', friends_file_name)
        if id_quali > 80:
            print("No file for id", id_quali)
            continue
        #parser les fichiers
        with open(friends_path) as friends_jsonl_file:
            for friends_line in friends_jsonl_file: 
                friends_json_data = json.loads(friends_line)
                #extraire les informations utiles pour la liste d'adjacence 
                friend_twitter_handle = friends_json_data['screen_name']
                friend_id = friends_json_data['id_str']
                #les stocker en paires dans le dictionnaire préalablement ouvert
                friend_name_per_id[friend_id] = friend_twitter_handle
                #écrire le contenu dans le fichier csv
                friends_edges_file.write('{},{}\n'.format(user_id, friend_id))
                                
with open(os_path.join(git_dir, 'Project_Files', 'friends_labels.csv'), 'w') as labels_file: #créer un nouveau fichier csv destiné à contenir la liste des ids et leur screen name correspondant
    for friend_id, friend_twitter_handle in friend_name_per_id.items(): #pour chaque élément du dictionnaire des paires de id:screen_name
        labels_file.write('{},{}\n'.format(friend_id, friend_twitter_handle)) #écrire le contenu du nouveau csv



Skipping duplicated user: yoyehudi
Skipping duplicated user: philipdurbin
Skipping duplicated user: matkuzak
Twitter id not found in quali file: Markussuhr
Skipping duplicated user: emmy_ft
No file for id 81
No file for id 83
No file for id 84
No file for id 85


## Créer les fichiers de liens et de noeuds pour le réseau des followers

In [16]:
follower_name_per_id = {} # créer un dictionnaire des paires de id:screen_name
#créer un nouveau fichier csv destiné à contenir la liste d'adjacence des liens
with open(os_path.join(git_dir, 'Project_Files', 'edges_followers.csv'), 'w') as edges_file : 
    for user in json_data:
        user_id = user['id_str']
        twitter_handle = user['screen_name']
        #créer une variable avec l'information qui sert à ordonner les utilisateurs
        try:
            id_quali = quali_df.loc[twitter_handle, 'INDEX']
        except:
            print("Twitter id not found in quali file:", twitter_handle)#indiquer les données manquantes
        if id_quali.size != 1:
            print("Skipping duplicated user:", id_quali.index[0])#indiquer les doublons
            continue
        assert id_quali > 0 and id_quali <= 85
        #Ouvrir les fichiers en fonction de leur identifiant quali
        followers_file_name = 'follower_ids{}.txt'.format(id_quali)
        followers_path = os_path.join(git_dir, 'twarc_data', 'followers', followers_file_name)
        if id_quali > 80:
            print("No file for id", id_quali)
            continue
        for follower_id in open(followers_path):
            #extraire les informations utiles pour la liste d'adjacence 
            follower_id = follower_id.strip()#enlever la ligne de trop
            #écrire le contenu dans le fichier csv
            edges_file.write('{},{}\n'.format(user_id, follower_id))
            
with open(os_path.join(git_dir, 'Project_Files', 'followers_labels.csv'), 'w') as followers_labels_file: #créer un nouveau fichier csv destiné à contenir la liste des ids et leur screen name correspondant
    for follower_id, follower_twitter_handle in follower_name_per_id.items(): #pour chaque élément du dictionnaire des paires de id:screen_name
        followers_labels_file.write('{},{}\n'.format(follower_id, follower_twitter_handle)) #écrire le contenu du nouveau csv

Skipping duplicated user: yoyehudi
Skipping duplicated user: philipdurbin
Skipping duplicated user: matkuzak
Twitter id not found in quali file: Markussuhr
Skipping duplicated user: emmy_ft
No file for id 81
No file for id 83
No file for id 84
No file for id 85
